# 03 — Training STARK-ST (base training)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/balk21/stark-cls-finetune/blob/main/notebooks/03_train.ipynb)

Trains STARK-ST with the **original STARK procedure** (same sampling, augmentation, losses, optimizer, learning
rates, schedule and samples per epoch), on **any combination of datasets** (currently prepared automatically:
GOT-10k and COCO 2017). Works on Google Colab, Vast.ai and a local machine.

| Stage | Trains | Epochs (LR ÷10 at) | Loss | Starts from |
|---|---|---|---|---|
| 1 | backbone + transformer + box head | 500 (400) | GIoU ×2 + L1 ×5 | ImageNet backbone |
| 2 | classification (score) head only | 50 (40) | BCE | stage-1 weights (`init`) |

Both stages: 60 000 samples per epoch, AdamW (lr 1e-4, backbone ×0.1, weight decay 1e-4), gradient clipping 0.1.
The original was trained on 8 GPUs × 16 samples = **128 samples per optimizer step**; here the same effective batch is
reached on one GPU by **gradient accumulation** (`effective_batch=128`, `micro_batch=16` → 8 passes per step). This is
mathematically the same update (the backbone BatchNorm layers are frozen in STARK).

**Time.** Stage 2 reads 3 M samples, stage 1 30 M. Measured on an RTX 3060 laptop GPU: stage 2 ≈ 35 samples/s
(≈ 24 h), stage 1 ≈ 20 samples/s (≈ 17 days). Faster GPUs (A100, L4) are several times faster; the progress line
shows samples/s and the remaining time, so start a run and read the estimate after the first steps.
On Colab, prefer an **A100** (or L4) runtime: they use TF32 like the RTX 3000/4000 series; a T4 does not.

**Interruptions are fine.** After every epoch a complete checkpoint is written (on Colab to Google Drive); running
the same parameters again **resumes** from the last finished epoch, bit-identical to an uninterrupted run.

| Dataset | Source | Archive size | Notes |
|---|---|---|---|
| COCO 2017 train | downloaded automatically from images.cocodataset.org | 19.6 GB | 118 287 images |
| GOT-10k train | needs a free registration: [got-10k.aitestunion.com/downloads](http://got-10k.aitestunion.com/downloads); the links come by e-mail (Google Drive links: see the comment in step 1) | `full_data.zip`: 70.7 GB | 9 335 videos; the `vottrain` split (VOT-overlapping videos removed) is used for training, `votval` for validation |

The archives are kept (on Colab: on Google Drive in `MyDrive/LOKAP/train_archives/`) and extracted to
the local disk in every new session. The local disk must hold the extracted datasets; if it is too small, the
preparation stops with a clear message.

In [ ]:
# Run this cell once at the top. On Google Colab it clones / updates the repository; the session itself
# (environment, checkpoint, paths) is prepared in step 2.
DRIVE_ROOT = "/content/drive/MyDrive/LOKAP"   # Colab only: Google Drive folder for everything (same in all notebooks)

import sys, subprocess
from pathlib import Path
if "google.colab" in sys.modules:
    REPO = Path("/content/stark-cls-finetune")
    if not REPO.exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/balk21/stark-cls-finetune.git", str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "pull", "-q", "--ff-only"])   # get the latest fixes
    sys.path.insert(0, str(REPO / "notebooks"))
else:
    sys.path.insert(0, str(Path.cwd()))   # the notebooks/ folder (nbhelper.py lives here)
import importlib, nbhelper as nb
nb = importlib.reload(nb)   # use the current code, also when this cell is run again after a git pull

## 1. Parameters

README section 4.2 shows which lines to change for each training; every parameter is explained in
`docs/details.md` (section 7.4). `None` = the original STARK value.
The run folder name is generated from the parameters (e.g. `st101_stage1_coco_s42`) unless `name` is set.

In [ ]:
TRAIN = dict(
    name=None,                     # run folder name; None = generated from the parameters

    # --- What to train ---
    model_config="baseline_R101",  # "baseline_R101" (STARK-ST101) | "baseline" (STARK-ST50)
    stage=1,                       # 1 = backbone + transformer + box head (from ImageNet), 2 = classification head
    init=None,                     # stage 1: None (ImageNet backbone). Stage 2 (required): the finished stage-1 run on the
                                   #   same datasets, e.g. "st101_stage1_coco_s42", or "official" (STARK's weights,
                                   #   trained on all four datasets; see README section 4.2)

    # --- Data ---
    datasets=["got10k", "coco"],   # any non-empty combination of "got10k", "coco" ("lasot", "trackingnet": manual)
    dataset_ratios=None,           # sampling weight per dataset; None = equal (as in STARK)
    val_datasets=["got10k"],       # validation on the GOT-10k votval split; [] = no validation (no GOT-10k needed)

    # --- Schedule (None = original value) ---
    epochs=None,                   # stage 1: 500, stage 2: 50
    lr_drop_epoch=None,            # stage 1: 400, stage 2: 40
    samples_per_epoch=None,        # 60000
    val_samples_per_epoch=None,    # 10000
    val_interval=None,             # validate every N epochs (stage 1: 20, stage 2: 10)

    # --- Batch / machine ---
    effective_batch=128,           # samples per optimizer step; 128 = original (8 GPUs x 16)
    micro_batch=16,                # samples per forward/backward pass; lower it if the GPU runs out of memory
    num_workers=8,                 # data loading processes (lower it if the runtime runs out of RAM)
    seed=42,                       # STARK default
    keep_every=None,               # keep the weights every N epochs (None: stage 1: 50, stage 2: 10)
)

# GOT-10k archives: download links from the registration e-mail, or paths of archives.
# On Colab, if the link is a Google Drive file (e.g. full_data.zip), do NOT put it here: shared Drive files often hit
# Google's daily download limit ("Quota exceeded"). Instead, in the browser: open the link -> "Add shortcut to Drive"
# -> right-click the shortcut -> "Make a copy" -> move the copy into <DRIVE_ROOT>/train_archives/got10k/, and leave
# this list empty. Your own copy is read directly from Drive (README, section 4.1).
GOT10K_URLS = []

## 2. Prepare the session and the data, check the parameters

On Colab: mounts Google Drive and restores the environment and the checkpoint (as in `00_setup_colab.ipynb`).
Then downloads / extracts the datasets (only what is missing) and shows what the run will do.
The **first** time, downloading COCO takes a while (19.6 GB); later sessions only extract the archives.

In [ ]:
info = nb.setup_training(TRAIN, got10k_urls=GOT10K_URLS, drive_root=DRIVE_ROOT)

## 3. Train

The output is streamed here (samples/s, epoch and training ETA, losses). Stopping the cell (■) stops the run;
running this cell again resumes from the last finished epoch. On Colab the run folder is on Google Drive
(`MyDrive/LOKAP/training/<run>/`), so a new session can resume it after step 2.

| File in the run folder | Content |
|---|---|
| `train_config.json` | all parameters, dataset folders, initial weights, GPU, code version |
| `history.csv`, `history.png` | per-epoch losses (train / validation), learning rate, duration |
| `logs/train.log` | the progress output |
| `checkpoints/latest.pth.tar` | complete state after the last epoch (for resuming) |
| `checkpoints/STARKST_epXXXX.pth.tar` | weights every `keep_every` epochs and at the end |
| `final.pth.tar` | final weights (written when the run finishes) |

In [ ]:
RUN_DIR = nb.train(TRAIN)

## 4. Progress / losses

Can be run at any time for a running or finished run (in a new session, run steps 1 and 2 first).

In [ ]:
nb.show_training(TRAIN)          # or nb.show_training("<run name>")

In [ ]:
nb.list_training_runs()

## 5. Using the trained model

**Evaluate a stage-2 run** in `01_run_experiment.ipynb` with
`checkpoint="train:<run name>"` (and the same `model_config`), e.g.
`checkpoint="train:st101_stage2_coco_from-st101_stage1_coco_s42_s42"`.
`01_run_experiment.ipynb` downloads (on Colab: or restores from Drive) the sequences it evaluates on.

**Stage 1 → stage 2:** after a stage-1 run has finished, train stage 2 on top of it, on the same datasets
(stage 2 always needs `init`; each run has its own folder, named after stage, datasets and `init`):
```python
TRAIN_STAGE2 = dict(TRAIN, stage=2, init="st101_stage1_coco_s42")   # the stage-1 run name printed by step 3
```

**Quick test of the whole pipeline** (a few minutes): `dict(TRAIN, epochs=2, samples_per_epoch=1024,
val_samples_per_epoch=256, val_interval=1, name="test_run")`. The numbers are meaningless, but data loading,
checkpointing, resuming and the export are exercised.

**Vast.ai / local machine:** the same notebook works. The datasets go to `train_data` (default `data/train/`, see
`configs/paths.yaml`) and the archives to `<train_data>/_archives/`. A GOT-10k copy that is already extracted can be
used directly: set `train_data` so that `<train_data>/got10k/train/list.txt` exists.